In [ ]:
#| default_exp graph

## Notebook symbol graph

Statically match notebook definitions and calls so agents can see which cells define, call, and depend on a symbol.

This notebook adds a static map across notebooks. It does not execute project code; it parses cells with `ast`, records where symbols are defined, and reports which cells call those symbols.

That gives agents a fast way to answer questions like "where is this helper used?" before editing a private function or moving code between notebooks.

The graph is a navigation aid, not a runtime dependency analyzer. It answers practical maintenance questions: where is this symbol defined, who calls it, and are any notebooks importing private helpers that should stay local?

```python
symbol_graph(path="nbs", symbol="write_nb")
private_symbol_report(path="nbs")
```

In [0]:
import tempfile as _tempfile
from pathlib import Path as _Path
from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb, write_nb as _write_nb
from nbskill.graph import symbol_graph

with _tempfile.TemporaryDirectory() as td:
    root = _Path(td) / "nbs"
    root.mkdir()
    _write_nb(_new_nb([
        _mk_cell("#| default_exp demo"),
        _mk_cell("#| export\ndef helper():\n    return 1\n\ndef target():\n    return helper()"),
        _mk_cell("target()"),
    ]), root / "demo.ipynb")
    symbol_graph(str(root), "helper")

Symbol helper
Definitions:
- /var/folders/6_/45pyyxdd7hz3wz33p813bx_c0000gn/T/tmpg_12_ioy/nbs/demo.ipynb id=17728736
Callers:
- /var/folders/6_/45pyyxdd7hz3wz33p813bx_c0000gn/T/tmpg_12_ioy/nbs/demo.ipynb id=17728736
Callees:
- (none)


In [ ]:
#| export
import ast
import glob
from pathlib import Path

from fastcore.nbio import read_nb as _read_nb
from fastcore.script import call_parse

from nbskill.foundation import cell_source, cli_error, cli_return, is_export_directive, tracked_call

### Finding notebooks and removing directives

The graph starts with notebook paths. These helpers accept a file, folder, or glob, skip checkpoint folders, and remove nbdev directives before parsing Python so `ast` sees regular source.

In [ ]:
#| export
def _source_without_directives(source):
    return "\n".join(line for line in source.splitlines() if not is_export_directive(line))


def _is_notebook_path(path):
    path = Path(path)
    return path.suffix == ".ipynb" and ".ipynb_checkpoints" not in path.parts


def _notebook_paths(path="nbs"):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"):
        candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir():
        candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file():
        candidates = [pth]
    else:
        candidates = []
    paths = sorted({path for path in candidates if _is_notebook_path(path)})
    if not paths: cli_error(f"No notebooks matched {path!r}")
    return paths


def _graph_scope(path):
    pth = Path(str(path)).expanduser()
    return pth.parent if pth.is_file() else path

### Discovering definitions and calls

The parser records function, class, and method definitions, then walks call expressions inside each parsed tree. Both fully qualified names and short names are kept because notebooks often import helpers into local scope.

In [ ]:
#| export
def _parse_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: return ast.parse(_source_without_directives(cell_source(cell)))
    except SyntaxError: return None


def _name_parts(node):
    if isinstance(node, ast.Name): return [node.id]
    if isinstance(node, ast.Attribute):
        base = _name_parts(node.value)
        return [*base, node.attr] if base else [node.attr]
    return []


def _call_name(node):
    parts = _name_parts(node)
    return ".".join(parts) if parts else None


def _call_names(node):
    names = []
    for child in ast.walk(node):
        if not isinstance(child, ast.Call): continue
        name = _call_name(child.func)
        if not name: continue
        names.append(name)
        short = name.rsplit(".", 1)[-1]
        if short != name: names.append(short)
    return tuple(dict.fromkeys(names))


def _node_definitions(node):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return [(node.name, node, "function")]
    if isinstance(node, ast.ClassDef):
        items = [(node.name, node, "class")]
        for child in node.body:
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                items.append((f"{node.name}.{child.name}", child, "method"))
        return items
    return []

### Building graph records

Each code cell can contribute definition records and caller records. The collected graph is deliberately simple dictionaries so reports, tests, and future tools can inspect it without a graph database.

In [ ]:
#| export
def _cell_definition_records(path, module, idx, cell):
    tree = _parse_cell(cell)
    if tree is None: return []
    records = []
    for node in tree.body:
        for symbol, symbol_node, kind in _node_definitions(node):
            records.append({
                "symbol": symbol,
                "kind": kind,
                "module": module,
                "path": str(path),
                "cell_id": getattr(cell, "id", ""),
                "cell_idx": idx,
                "calls": _call_names(symbol_node),
            })
    return records


def _cell_call_record(path, idx, cell):
    tree = _parse_cell(cell)
    if tree is None: return None
    calls = _call_names(tree)
    if not calls: return None
    return {"path": str(path), "cell_id": getattr(cell, "id", ""), "cell_idx": idx, "calls": calls}


def _import_module_name(node):
    if node.module is None: return None
    if node.module == "nbskill": return ""
    if node.module.startswith("nbskill."): return node.module.removeprefix("nbskill.")
    return node.module


def _cell_import_records(path, idx, cell):
    tree = _parse_cell(cell)
    if tree is None: return []
    records = []
    for node in ast.walk(tree):
        if not isinstance(node, ast.ImportFrom): continue
        module = _import_module_name(node)
        if module is None: continue
        for alias in node.names:
            records.append({
                "module": module,
                "symbol": alias.name,
                "local": alias.asname or alias.name,
                "path": str(path),
                "cell_id": getattr(cell, "id", ""),
                "cell_idx": idx,
            })
    return records


def _notebook_module_name(path, nb):
    for cell in nb.cells:
        for line in cell_source(cell).splitlines():
            line = line.strip()
            if line.startswith("#| default_exp "):
                return line.split(None, 2)[-1].replace("/", ".")
    return Path(path).stem


def _collect_graph(path="nbs"):
    definitions, callers, imports = [], [], []
    for nb_path in _notebook_paths(path):
        nb = _read_nb(nb_path)
        module = _notebook_module_name(nb_path, nb)
        for idx, cell in enumerate(nb.cells):
            definitions.extend(_cell_definition_records(nb_path, module, idx, cell))
            imports.extend(_cell_import_records(nb_path, idx, cell))
            record = _cell_call_record(nb_path, idx, cell)
            if record: callers.append(record)
    return {"definitions": definitions, "callers": callers, "imports": imports}

### Resolving symbol relationships

The graph is approximate by design. Matching by exact or short symbol name is enough to surface likely callers and callees, which is the useful pre-edit signal for this project.

In [ ]:
#| export
def _symbol_short_name(symbol):
    return str(symbol).rsplit(".", 1)[-1]


def _call_matches_symbol(call, symbol):
    call = str(call)
    symbol = str(symbol)
    return call == symbol or _symbol_short_name(call) == _symbol_short_name(symbol)


def _definitions_for_symbol(graph, symbol):
    return [record for record in graph["definitions"] if record["symbol"] == symbol or _symbol_short_name(record["symbol"]) == symbol]


def _caller_records_for_symbol(graph, symbol):
    return [record for record in graph["callers"] if any(_call_matches_symbol(call, symbol) for call in record["calls"])]


def _resolve_callees(graph, calls):
    symbols = {record["symbol"] for record in graph["definitions"]}
    resolved = []
    for call in calls:
        for symbol in symbols:
            if _call_matches_symbol(call, symbol): resolved.append(symbol)
    return sorted(set(resolved))


def _locations(records):
    return [f"{record['path']} id={record['cell_id']}" for record in records]


def _callee_locations(graph, symbol):
    return _locations(_definitions_for_symbol(graph, symbol))

### Formatting reports

The reporting helpers turn graph records into compact text. They are meant for agent context: locations include notebook paths and cell ids so the next step can jump straight to `read_nb` or `show_doc`.

In [ ]:
#| export
def _symbol_graph_data(path, symbol):
    graph = _collect_graph(_graph_scope(path))
    definitions = _definitions_for_symbol(graph, symbol)
    callers = _caller_records_for_symbol(graph, symbol)
    callee_symbols = []
    for definition in definitions:
        callee_symbols.extend(_resolve_callees(graph, definition["calls"]))
    callee_symbols = sorted(set(item for item in callee_symbols if item != symbol))
    return {"symbol": symbol, "definitions": definitions, "callers": callers, "callees": callee_symbols, "graph": graph}


def _format_symbol_graph_data(data):
    lines = [f"Symbol {data['symbol']}"]
    lines.append("Definitions:")
    lines.extend([f"- {loc}" for loc in _locations(data["definitions"]) ] or ["- (none)"])
    lines.append("Callers:")
    lines.extend([f"- {loc}" for loc in _locations(data["callers"]) ] or ["- (none)"])
    lines.append("Callees:")
    if data["callees"]:
        for symbol in data["callees"]:
            locs = "; ".join(_callee_locations(data["graph"], symbol)) or "unknown location"
            lines.append(f"- {symbol}: {locs}")
    else:
        lines.append("- (none)")
    return "\n".join(lines)


def symbol_usage_summary(path, symbols):
    "Return a compact caller/callee summary for one or more symbols."
    if isinstance(symbols, str): symbols = [symbols]
    chunks = []
    for symbol in dict.fromkeys(symbols or []):
        data = _symbol_graph_data(path, symbol)
        callers = "; ".join(_locations(data["callers"])[:8]) or "none"
        callees = "; ".join(data["callees"][:8]) or "none"
        chunks.append(f"{symbol}: callers={callers}; callees={callees}")
    return "\n".join(chunks)

### Public graph reports

`symbol_graph` focuses on one symbol's definitions, callers, and callees. `private_symbol_report` looks for cross-notebook calls to private helpers, which is useful when deciding whether a private function can be changed safely.

In [ ]:
#| export
@call_parse
@tracked_call
def symbol_graph(
    path: str = "nbs",  # Notebook file, directory, or glob to scan
    symbol: str = "",  # Function, class, or Class.method to inspect
):
    "Print definitions, callers, and callees for a notebook symbol."
    if not symbol: cli_error("Pass --symbol to inspect")
    text = _format_symbol_graph_data(_symbol_graph_data(path, symbol))
    print(text)
    return cli_return(text)


@call_parse
@tracked_call
def private_symbol_report(
    path: str = "nbs",  # Notebook file, directory, or glob to scan
):
    "Print cross-notebook calls to imported private `_` symbols."
    graph = _collect_graph(path)
    definitions = {(record.get("module"), record["symbol"]): record for record in graph["definitions"]}
    lines = ["Cross-notebook private symbol calls"]
    for imported in graph["imports"]:
        symbol = imported["symbol"]
        if not _symbol_short_name(symbol).startswith("_"): continue
        definition = definitions.get((imported["module"], symbol))
        if not definition or definition["path"] == imported["path"]: continue
        for caller in graph["callers"]:
            if caller["path"] != imported["path"]: continue
            if not any(_call_matches_symbol(call, imported["local"]) for call in caller["calls"]): continue
            lines.append(
                f"- {symbol} defined {definition['path']} id={definition['cell_id']} "
                f"called from {caller['path']} id={caller['cell_id']}"
            )
    if len(lines) == 1: lines.append("No cross-notebook private symbol calls found.")
    text = "\n".join(dict.fromkeys(lines))
    print(text)
    return cli_return(text)

In [ ]:
import tempfile as _tempfile
from pathlib import Path as _Path

from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb
from fastcore.nbio import write_nb as _write_tmp_nb
from nbskill.graph import private_symbol_report, symbol_graph, symbol_usage_summary
from nbskill.mcp import capture_call

with _tempfile.TemporaryDirectory() as td:
    root = _Path(td)
    lib = root / "lib.ipynb"
    caller = root / "caller.ipynb"
    _write_tmp_nb(_new_nb([
        _mk_cell("#| default_exp lib"),
        _mk_cell("#| export\ndef helper():\n    return 1\n\ndef _secret():\n    return helper()\n\ndef target():\n    return helper()"),
    ]), lib)
    _write_tmp_nb(_new_nb([
        _mk_cell("from nbskill.lib import target, _secret\nvalue = target()\n_secret()"),
    ]), caller)
    text = capture_call(symbol_graph, path=str(root), symbol="target")
    assert "Definitions:" in text
    assert "caller.ipynb id=" in text
    assert "helper" in text
    summary = symbol_usage_summary(str(root), ["target"])
    assert "callers=" in summary and "caller.ipynb id=" in summary
    report = capture_call(private_symbol_report, path=str(root))
    assert "_secret" in report
    assert "called from" in report